# Statistics for ML Quality Engineering: Confidence Intervals & Drift Detection

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset) — 569 real patient samples, 30 numeric features computed from digitized images of fine needle aspirate (FNA) cell nuclei, binary target (malignant / benign). Loaded via `sklearn.datasets.load_breast_cancer`.

## Problem statement

You've just trained a logistic regression classifier to flag malignant tumors. Before anyone signs off on deploying it, your job as the ML quality engineer is to answer three questions a single accuracy number can't:

1. **How uncertain is our performance estimate?** A single test-set AUC is a point estimate from one random split — it could easily be lucky or unlucky. You need a defensible *range*, not a single number.
2. **Does the model behave consistently across patient subgroups?** If the model performs noticeably worse on one subgroup than another, that's a quality/fairness risk worth flagging before deployment, not after.
3. **Would we notice if the incoming data started drifting?** In production, the features a monitoring pipeline sees next month won't look exactly like the training data. You need an automated statistical check that flags *which* features have shifted, not just a vague "something changed."

## What you should produce

By the end of this notebook you should have working code for:

- `bootstrap_ci(y_true, y_proba, metric_fn, n_boot, ci)` — a bootstrap resampling function that returns a confidence interval for any metric (not just a single point estimate), applied here to ROC-AUC and to accuracy.
- `subgroup_metric_test(y_true, y_proba, groups, metric_fn)` — a function that splits predictions by a subgroup label (we'll use tumor size, split at the median `mean radius`), computes the metric per subgroup, and runs a two-sample statistical test (bootstrap difference-in-metric test) to say whether the gap is statistically significant or just noise.
- `drift_report(reference_df, current_df, alpha)` — a function that runs a Kolmogorov–Smirnov two-sample test feature-by-feature between a "reference" batch and a "current" batch of real data, and returns a table flagging exactly which features drifted at a given significance level, with a multiple-comparisons correction (since you're running ~30 simultaneous hypothesis tests).

Each function should work on the real dataset provided below — no synthetic data needed, since the dataset already has enough natural subgroup and batch structure to exercise all three checks honestly.


## Setup

Run this cell as-is. It loads the real dataset, splits it into a train and a held-out test set, and fits a baseline logistic regression model you'll evaluate below.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score

RNG = np.random.default_rng(42)

data = load_breast_cancer(as_frame=True)
X = data.frame[data.feature_names]
y = data.frame["target"]  # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=5000, random_state=42).fit(X_train_s, y_train)
y_proba = clf.predict_proba(X_test_s)[:, 1]
y_pred = clf.predict(X_test_s)

print(f"Test AUC:      {roc_auc_score(y_test, y_proba):.4f}")
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")


## Part 1 — Bootstrap confidence interval for a metric

A single AUC/accuracy number from one test split is a point estimate. Fill in `bootstrap_ci` below: it should resample `(y_true, y_proba)` pairs **with replacement** `n_boot` times, compute `metric_fn` on each resample, and return the point estimate plus the lower/upper bounds of the `ci` confidence interval (use the percentile method).

Use it to report a 95% CI for both ROC-AUC and accuracy on the test set.

In [ ]:
def bootstrap_ci(y_true, y_proba, metric_fn, n_boot=2000, ci=0.95, random_state=42):
    """
    Bootstrap confidence interval for a classification metric.

    Parameters
    ----------
    y_true : array-like of shape (n,) -- ground-truth labels
    y_proba : array-like of shape (n,) -- predicted probabilities (or scores/labels,
              whatever metric_fn expects as its second argument)
    metric_fn : callable(y_true, y_proba) -> float
    n_boot : int -- number of bootstrap resamples
    ci : float -- confidence level, e.g. 0.95 for a 95% CI

    Returns
    -------
    (point_estimate, lower_bound, upper_bound)
    """
    # TODO: implement bootstrap resampling here
    raise NotImplementedError


# TODO: call bootstrap_ci for roc_auc_score and for accuracy_score (on y_pred),
# and print the point estimate + CI for each, e.g.:
# auc_point, auc_lo, auc_hi = bootstrap_ci(y_test.values, y_proba, roc_auc_score)
# print(f"AUC: {auc_point:.4f}  95% CI [{auc_lo:.4f}, {auc_hi:.4f}]")


## Part 2 — Subgroup consistency check

Tumor size (`mean radius`) is a real, clinically meaningful feature. Split the **test set** into "small" and "large" tumor subgroups at the median `mean radius`, compute AUC separately for each subgroup, and test whether the difference is statistically significant (not just sampling noise) using a bootstrap test on the *difference* in AUC between subgroups.

Fill in `subgroup_metric_test` below.

In [ ]:
def subgroup_metric_test(y_true, y_proba, group_mask, metric_fn, n_boot=2000, random_state=42):
    """
    Compare metric_fn between two subgroups defined by a boolean mask,
    and test whether the difference is statistically significant via bootstrap.

    Parameters
    ----------
    y_true : array-like of shape (n,)
    y_proba : array-like of shape (n,)
    group_mask : boolean array of shape (n,) -- True = group A, False = group B
    metric_fn : callable(y_true, y_proba) -> float
    n_boot : int

    Returns
    -------
    dict with keys: metric_a, metric_b, diff, p_value
        p_value is a two-sided bootstrap p-value for H0: diff == 0
    """
    # TODO: implement
    raise NotImplementedError


# TODO: build group_mask from X_test["mean radius"] vs its median, then call
# subgroup_metric_test with roc_auc_score and print metric_a, metric_b, diff, p_value.


## Part 3 — Drift detection across real data batches

In production you'd compare a "reference" batch (e.g. training-time data) against a "current" batch (e.g. this week's incoming data) feature by feature. Here, simulate that honestly using **real** data: treat `X_train` as the reference batch and `X_test` as the current batch (they're genuinely different random samples of the same real population — a reasonable stand-in for "did anything drift between the batch we validated on and the batch we're scoring now?").

Fill in `drift_report`: for every feature, run a two-sample Kolmogorov–Smirnov test (`scipy.stats.ks_2samp`) comparing its distribution in the reference vs. current batch. Because you're running ~30 tests at once, apply a Bonferroni correction to `alpha` before flagging a feature as drifted. Return a DataFrame sorted by p-value with columns `feature`, `ks_stat`, `p_value`, `drifted`.

In [ ]:
def drift_report(reference_df, current_df, alpha=0.05):
    """
    Per-feature KS-test drift report between two real data batches.

    Parameters
    ----------
    reference_df : DataFrame of shape (n_ref, n_features)
    current_df : DataFrame of shape (n_cur, n_features), same columns as reference_df
    alpha : float -- family-wise significance level before Bonferroni correction

    Returns
    -------
    DataFrame with columns: feature, ks_stat, p_value, drifted
    sorted by p_value ascending. `drifted` is True if p_value < alpha / n_features.
    """
    # TODO: implement
    raise NotImplementedError


# TODO: call drift_report(X_train, X_test) and display the result.
# Sanity check: since X_train/X_test are a random split of the SAME population,
# you should see few (ideally zero) features flagged as drifted -- that's the
# correct behavior for a drift detector, not a bug!


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

**Part 1 — Bootstrap CI.** Resample indices with replacement, recompute the metric each time, and take percentiles of the resulting distribution. This directly approximates the sampling distribution of the metric without assuming normality.

```python
def bootstrap_ci(y_true, y_proba, metric_fn, n_boot=2000, ci=0.95, random_state=42):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    point = metric_fn(y_true, y_proba)
    boot_stats = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_stats[i] = metric_fn(y_true[idx], y_proba[idx])
    alpha = (1 - ci) / 2
    lo, hi = np.quantile(boot_stats, [alpha, 1 - alpha])
    return point, lo, hi


auc_point, auc_lo, auc_hi = bootstrap_ci(y_test.values, y_proba, roc_auc_score)
print(f"AUC:      {auc_point:.4f}  95% CI [{auc_lo:.4f}, {auc_hi:.4f}]")

acc_point, acc_lo, acc_hi = bootstrap_ci(y_test.values, y_pred, accuracy_score)
print(f"Accuracy: {acc_point:.4f}  95% CI [{acc_lo:.4f}, {acc_hi:.4f}]")
```

Note `metric_fn(y_true[idx], y_proba[idx])` works for both AUC (needs scores) and accuracy (needs hard labels) since we pass in whichever array is appropriate as `y_proba`.

**Part 2 — Subgroup test.** Compute the metric on each subgroup, then bootstrap the *difference* by resampling within each subgroup independently and recomputing the gap many times. The p-value is the fraction of bootstrap differences at least as extreme as zero would need to be, under a two-sided test centered on the observed difference (equivalently: how often a bootstrap gap crosses zero relative to the observed sign).

```python
import warnings
from sklearn.exceptions import UndefinedMetricWarning

def subgroup_metric_test(y_true, y_proba, group_mask, metric_fn, n_boot=2000, random_state=42):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    group_mask = np.asarray(group_mask)

    yt_a, yp_a = y_true[group_mask], y_proba[group_mask]
    yt_b, yp_b = y_true[~group_mask], y_proba[~group_mask]

    metric_a = metric_fn(yt_a, yp_a)
    metric_b = metric_fn(yt_b, yp_b)
    observed_diff = metric_a - metric_b

    n_a, n_b = len(yt_a), len(yt_b)
    boot_diffs = np.empty(n_boot)
    with warnings.catch_warnings():
        # a bootstrap resample of a near-perfectly-separable subgroup can
        # occasionally contain only one class -- AUC is undefined there, so
        # we drop those resamples below rather than let them skew the p-value
        warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
        for i in range(n_boot):
            idx_a = rng.integers(0, n_a, n_a)
            idx_b = rng.integers(0, n_b, n_b)
            boot_diffs[i] = metric_fn(yt_a[idx_a], yp_a[idx_a]) - metric_fn(yt_b[idx_b], yp_b[idx_b])

    boot_diffs = boot_diffs[~np.isnan(boot_diffs)]

    # two-sided p-value: fraction of bootstrap diffs at least as far from 0
    # as the observed diff is from the "no effect" null, using the bootstrap
    # distribution's spread around its own mean (standard bootstrap hypothesis test)
    centered = boot_diffs - boot_diffs.mean()
    p_value = np.mean(np.abs(centered) >= np.abs(observed_diff))

    return {"metric_a": metric_a, "metric_b": metric_b, "diff": observed_diff, "p_value": p_value}


group_mask = (X_test["mean radius"] > X_test["mean radius"].median()).values
result = subgroup_metric_test(y_test.values, y_proba, group_mask, roc_auc_score)
print(f"AUC (large tumors): {result['metric_a']:.4f}")
print(f"AUC (small tumors): {result['metric_b']:.4f}")
print(f"Difference: {result['diff']:.4f}   p-value: {result['p_value']:.4f}")
```

With real data you'll typically find the AUC for the "large tumor" subgroup is high but compressed (most large tumors are confidently malignant, little ranking information left), while small tumors — where the decision is genuinely harder — show more spread. Whether the gap clears p < 0.05 depends on the random seed, which is itself the point: **this is exactly the kind of check that should run automatically, because "looks fine on the overall metric" can hide a real subgroup gap.**

**Part 3 — Drift report.** Run `ks_2samp` per column, divide alpha by the number of features tested (Bonferroni), and flag anything under the corrected threshold.

```python
def drift_report(reference_df, current_df, alpha=0.05):
    rows = []
    n_features = reference_df.shape[1]
    corrected_alpha = alpha / n_features
    for col in reference_df.columns:
        ks_stat, p_value = stats.ks_2samp(reference_df[col], current_df[col])
        rows.append({"feature": col, "ks_stat": ks_stat, "p_value": p_value})
    report = pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)
    report["drifted"] = report["p_value"] < corrected_alpha
    return report


report = drift_report(X_train, X_test)
display(report)
print(f"Features flagged as drifted: {report['drifted'].sum()} / {len(report)}")
```

Since `X_train`/`X_test` are a random split of the *same* underlying population, the honest result is that **zero (or very few) features should be flagged** after the Bonferroni correction — the detector correctly reports "no drift" on non-drifting data. This matters as much as catching real drift: a detector that fires on every feature of a normal train/test split is useless in production because it cries wolf constantly. If you re-run `drift_report` comparing, say, only-malignant vs. only-benign rows instead, you'll see most features correctly flagged, since that split *is* a real distributional shift.

</details>
